# Build an ML engineer agent with Claude Managed Agents

## Introduction

A data scientist hands over a model in a notebook; an ML engineer turns it into something a team can deploy. In this cookbook you'll build an agent that takes a dataset and a model spec and returns production scaffolding: a reproducible training pipeline, a serialized model with a schema-validated inference function, a FastAPI serving stub, a Dockerfile, and passing tests.

You'll run it on [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview), Anthropic's hosted runtime for stateful, tool-using agents, built on four core concepts:

- **Agent**: the model, system prompt, tools, MCP servers, and skills
- **Environment**: a configured container template (packages, network access)
- **Session**: a running agent instance within an environment, performing a specific task and generating outputs
- **Events**: messages exchanged between your application and the agent (user turns, tool results, status updates)

### What you'll learn

By the end of this cookbook you'll be able to:

- Write a system prompt focused on production concerns: reproducibility, input validation, latency, tests
- Have the agent self-verify its deliverable by running its own test suite before finishing
- Retrieve a deployable project (code + model artifact + Dockerfile) from a session

### Prerequisites

- Python 3.11+
- An Anthropic API key from the [Console](https://platform.claude.com/settings/keys), set as `ANTHROPIC_API_KEY`

Install dependencies:

In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [ ]:
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv, set_key

load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-4-6"

## 1. Create an environment

The environment carries the training stack plus the serving stack: `scikit-learn` and `pandas` for the pipeline, `fastapi` and `httpx` so the agent can spin up and integration-test its own API, and `pytest` for the suite. Networking is `unrestricted` for simplicity; in production, allowlist only what the build needs.

In [ ]:
env = client.beta.environments.create(
    name="cookbook-ml-engineer-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        "packages": {
            "type": "packages",
            "pip": [
                "pandas",
                "numpy",
                "scikit-learn",
                "fastapi",
                "uvicorn",
                "httpx",
                "pytest",
                "pydantic",
            ],
        },
    },
)

## 2. Create the agent

This system prompt cares about different things than the data scientist's: not squeezing out another point of AUC, but determinism, input validation, latency, and tests that actually run. The key line is the **self-verification requirement** — the agent must execute its own pytest suite and only report success if everything passes, which turns "looks plausible" into "demonstrated working".

Web tools are disabled; everything needed is preinstalled.

In [ ]:
ML_ENGINEER_SYSTEM_PROMPT = """\
You are a senior ML engineer producing deployable, tested model services.

## Pipeline rules
- One sklearn Pipeline end to end: preprocessing (ColumnTransformer) and
  model in a single object. random_state=42 everywhere. Save with joblib.
- A `train.py` that retrains from the CSV and reproduces metrics exactly.
- Validate inference inputs with a pydantic model; reject unknown
  categories and out-of-range values with clear error messages.

## Serving rules
- `app.py`: FastAPI with POST /predict (single + batch), GET /health,
  and GET /model-info (version, training date, metrics).
- Measure p50/p95 prediction latency over 200 calls with the TestClient
  and include the numbers in the README.

## Verification (mandatory)
- Write a pytest suite: pipeline determinism, schema rejection cases,
  endpoint contract tests via fastapi.testclient.
- Run `python3 -m pytest -q` and iterate until exit code 0. Never report
  success with failing tests; paste the final pytest summary line.

## Output (all to /mnt/session/outputs/)
- `service/` (train.py, app.py, schemas.py, model.joblib, metrics.json)
- `tests/` (test_pipeline.py, test_api.py)
- `Dockerfile` (slim Python base, non-root user, uvicorn entrypoint)
- `README.md` (run instructions, API examples, latency numbers)
Confirm "Saved: service + tests + Dockerfile" when done.
"""

agent = client.beta.agents.create(
    name="cookbook-ml-engineer",
    model=MODEL,
    system=ML_ENGINEER_SYSTEM_PROMPT,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
)

## 3. Generate and upload a dataset

We synthesize a house-price regression dataset with realistic feature types — numeric, categorical, and a date the pipeline must handle — so the generated service has real preprocessing to validate, not just a passthrough.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 600

sqft = rng.integers(450, 4200, n)
beds = rng.integers(1, 6, n)
city = rng.choice(["springfield", "shelbyville", "ogdenville"], n, p=[0.5, 0.3, 0.2])
year_built = rng.integers(1950, 2024, n)

city_premium = {"springfield": 1.0, "shelbyville": 1.25, "ogdenville": 0.8}
price = (
    sqft * 180
    + beds * 9000
    + (year_built - 1950) * 350
) * np.vectorize(city_premium.get)(city)
price = (price * rng.normal(1.0, 0.08, n)).round(-2)

df = pd.DataFrame(
    {
        "sqft": sqft,
        "bedrooms": beds,
        "city": city,
        "year_built": year_built,
        "price": price,
    }
)

DATA_PATH = Path("houses.csv")
df.to_csv(DATA_PATH, index=False)

with DATA_PATH.open("rb") as f:
    dataset = client.beta.files.upload(file=(DATA_PATH.name, f, "text/csv"))

print(f"Uploaded {DATA_PATH.name} ({dataset.size_bytes} bytes) as {dataset.id}")

## 4. Create a session and send the task

The task pins down the prediction target, the input schema the API must enforce, and the acceptance criterion (tests green). Everything else is delegated to the system prompt.

In [ ]:
MOUNT_PATH = f"/mnt/session/uploads/{DATA_PATH.name}"
RESOURCES = [{"type": "file", "file_id": dataset.id, "mount_path": MOUNT_PATH}]

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=RESOURCES,
    title="House price service",
)

TASK_PROMPT = f"""\
Build a price-prediction service from {MOUNT_PATH}.

Columns: sqft (int), bedrooms (int), city (one of springfield,
shelbyville, ogdenville), year_built (int), price (target, float).

The /predict endpoint must reject unknown cities and non-positive sqft.
Deliver the service/, tests/, Dockerfile, and README per your system
instructions, with the full pytest suite passing.
"""

client.beta.sessions.events.send(
    session.id,
    events=[
        {"type": "user.message", "content": [{"type": "text", "text": TASK_PROMPT}]},
    ],
)
print(f"Session {session.id} running")

## 5. Stream the run

Open the session in the [Console](https://platform.claude.com/) under **Sessions** to watch every event, tool call, and token count live. The helper below tails the same event stream, printing `agent.message` text and `agent.tool_use` calls as they arrive, and returns on `session.status_idle`.

In [ ]:
def wait_for_idle(session_id: str) -> None:
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            for block in ev.content:
                if block.type == "text":
                    text = block.text
                    print(text[:300] + ("..." if len(text) > 300 else ""))
        elif t in ("agent.tool_use", "agent.mcp_tool_use"):
            print(f"  [{ev.name}]")
        elif t == "session.status_idle":
            return
        elif t == "session.status_terminated":
            raise RuntimeError(
                "Session terminated before going idle. "
                f"Trace: https://platform.claude.com/sessions/{session_id}"
            )


wait_for_idle(session.id)

## 6. Retrieve the service

The download loop recreates the project tree locally — service code, model artifact, tests, Dockerfile, and README.

The [Files API](https://platform.claude.com/docs/en/api/beta/files/list) is a separate feature in beta, so to use `scope_id` here you also need to pass the Managed Agents beta header.

In [ ]:
outputs = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in outputs.data:
    print(f.filename, f.size_bytes)

INPUT_NAMES = {r["mount_path"].rsplit("/", 1)[-1] for r in RESOURCES}
downloaded = []
for f in outputs.data:
    if f.filename in INPUT_NAMES:
        continue
    content = client.beta.files.download(f.id)
    out_path = Path("price_service") / f.filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(content.read())
    downloaded.append(f.filename)
print("Downloaded:", downloaded)

if not any(name.endswith("Dockerfile") for name in downloaded):
    raise RuntimeError(f"Dockerfile not found. Files: {downloaded}")

## 7. Clean up and next steps

Archive the session and persist the IDs for reuse across future builds.

> **Warning:** make sure `.env` is listed in `.gitignore` before running the next cell — never commit it.

In [ ]:
client.beta.sessions.archive(session.id)

set_key(".env", "ML_ENGINEER_ENV_ID", env.id)
set_key(".env", "ML_ENGINEER_AGENT_ID", agent.id)
set_key(".env", "ML_ENGINEER_AGENT_VERSION", str(agent.version))
print("Saved ML_ENGINEER_ENV_ID, ML_ENGINEER_AGENT_ID, ML_ENGINEER_AGENT_VERSION to .env")

You've built an ML engineer agent that ships a deployable service: training pipeline, validated inference API, Docker packaging, and a test suite the agent ran itself before declaring victory.

From here:

- `cd price_service && python3 -m pytest -q` to confirm the suite passes on your machine.
- `docker build -t price-service . && docker run -p 8000:8000 price-service`, then POST to `/predict`.
- Chain it with the data scientist agent: have one session select the model, then feed its `metrics.json` and choices into this agent's task prompt.